In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import os

from torch.utils.data import Dataset, DataLoader


In [21]:
# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Using Apple Silicon GPU via MPS")
else:
    device = torch.device("cpu")
    print("⚠️ Using CPU only (no GPU acceleration available)")

✅ Using CUDA GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [20]:
import os
import cv2
import random
import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class ColorizationDataset(Dataset):
    def __init__(self, image_dir, image_size=32, max_size_gb=7):
        """
        image_dir: Folder containing all images (flat)
        image_size: Resize all images to this size (default 32 for CIFAR-10 compatibility)
        max_size_gb: Optional memory cap to limit dataset loading
        """
        self.image_paths = []
        total_size = 0
        max_size_bytes = max_size_gb * 1024**3

        # Collect image files
        all_files = [f for f in os.listdir(image_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not all_files:
            raise ValueError(f"No image files found in {image_dir}")

        random.shuffle(all_files)
        for f in all_files:
            path = os.path.join(image_dir, f)
            try:
                size = os.path.getsize(path)
            except OSError:
                continue
            if total_size + size > max_size_bytes:
                break
            self.image_paths.append(path)
            total_size += size

        print(f"✅ Loaded {len(self.image_paths)} images ({total_size / 1024**3:.2f} GB)")
        self.image_size = image_size

        # CIFAR-10-style transform (→ tensor, normalized to [-1, 1])
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # fallback to another random image if unreadable
            return self.__getitem__(random.randint(0, len(self.image_paths) - 1))

        img_tensor = self.transform(img)
        label = 0  # dummy label to keep (images, _) unpacking compatible
        return img_tensor, label


In [23]:
# Step 1: Dataset Preparation
# Use CIFAR-10 for simplicity (later replace with historical images)
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to tensors (0-1 range)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])
image_dir = r'/home/mmanani/ImageColourizationDataSet/FLAT_TRAIN'
dataset = ColorizationDataset(image_dir)

# Split into train/val/test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Function to convert RGB image to grayscale (for input)
def rgb_to_grayscale(img):
    # Convert tensor image to numpy, then to grayscale using OpenCV
    img_np = img.cpu().permute(1, 2, 0).numpy()  # Move to CPU before numpy
    img_np = (img_np * 0.5 + 0.5) * 255.0  # Denormalize to [0, 255]
    img_np = img_np.astype(np.uint8)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    gray = gray / 255.0  # Normalize to [0, 1]
    gray = torch.tensor(gray, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
    gray = gray * 2.0 - 1.0  # Normalize to [-1, 1]
    return gray.to(device)  # Move back to device

✅ Loaded 68567 images (7.00 GB)


In [24]:
# Step 2: Define the Generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            # Input: (batch_size, 1, 32, 32) grayscale
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),  # (64, 16, 16)
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # (128, 8, 8)
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # (64, 16, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),  # (3, 32, 32)
            nn.Tanh()  # Output: RGB image in [-1, 1]
        )

    def forward(self, x):
        return self.model(x)

In [25]:
# Step 3: Define the Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # Input: (batch_size, 3, 32, 32) RGB
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),  # (64, 16, 16)
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # (128, 8, 8)
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, kernel_size=8, stride=1, padding=0),  # (1, 1, 1)
            nn.Sigmoid()  # Output: Probability (real or fake)
        )

    def forward(self, x):
        return self.model(x)


In [26]:
# Step 4: Initialize Models, Loss, and Optimizers
generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCELoss()  # Binary Cross-Entropy for adversarial loss
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))


In [27]:
# Step 5: Training Loop
num_epochs = 20  # Small number for demo; increase for better results
real_label = 1.0
fake_label = 0.0

for epoch in range(num_epochs):
    epoch_start_time = time.time()  # Start timing the epoch
    total_images_processed = 0  # Track total images processed in epoch
    total_batch_time = 0.0  # Track total batch processing time

    for i, (images, _) in enumerate(dataloader):
        batch_start_time = time.time()  # Start timing the batch

        batch_size = images.size(0)
        images = images.to(device)

        # Create grayscale input
        grayscale_images = torch.stack([rgb_to_grayscale(img) for img in images]).to(device)

        # Train Discriminator
        discriminator.zero_grad()
        # Real images
        real_output = discriminator(images)
        real_loss = criterion(real_output.view(-1), torch.full((batch_size,), real_label, device=device))
        # Fake images
        fake_images = generator(grayscale_images)
        fake_output = discriminator(fake_images.detach())
        fake_loss = criterion(fake_output.view(-1), torch.full((batch_size,), fake_label, device=device))
        d_loss = real_loss + fake_loss
        d_loss.backward()
        d_optimizer.step()

        # Train Generator
        generator.zero_grad()
        fake_output = discriminator(fake_images)
        g_loss = criterion(fake_output.view(-1), torch.full((batch_size,), real_label, device=device))
        g_loss.backward()
        g_optimizer.step()

        # Calculate batch throughput
        batch_time = time.time() - batch_start_time  # Time for this batch
        batch_throughput = batch_size / batch_time if batch_time > 0 else 0  # Images per second
        total_images_processed += batch_size
        total_batch_time += batch_time

        if i % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{i}/{len(dataloader)}] "
                  f"D Loss: {d_loss.item():.4f} G Loss: {g_loss.item():.4f} "
                  f"Throughput: {batch_throughput:.2f} images/sec")

    # Calculate and print epoch throughput
    epoch_time = time.time() - epoch_start_time
    epoch_throughput = total_images_processed / epoch_time if epoch_time > 0 else 0
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Throughput: {epoch_throughput:.2f} images/sec")

    # Save sample images every epoch (unchanged from original code)
    with torch.no_grad():
        fake_images = generator(grayscale_images[:4]).cpu()
        grayscale_images = grayscale_images.cpu()
        images = images.cpu()

        # Denormalize for visualization
        fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)
        grayscale_images = (grayscale_images * 0.5 + 0.5).clamp(0, 1)
        images = (images * 0.5 + 0.5).clamp(0, 1)

        # Plot and save
        plt.figure(figsize=(12, 4))
        for j in range(4):
            plt.subplot(3, 4, j+1)
            plt.imshow(grayscale_images[j].squeeze(), cmap='gray')
            plt.title("Grayscale")
            plt.axis('off')
            plt.subplot(3, 4, j+5)
            plt.imshow(fake_images[j].permute(1, 2, 0))
            plt.title("Colorized")
            plt.axis('off')
            plt.subplot(3, 4, j+9)
            plt.imshow(images[j].permute(1, 2, 0))
            plt.title("Original")
            plt.axis('off')
        plt.savefig(f"output_epoch_{epoch+1}.png")
        plt.close()

Epoch [1/20] Batch [0/1072] D Loss: 1.3813 G Loss: 0.7028 Throughput: 113.21 images/sec
Epoch [1/20] Batch [100/1072] D Loss: 0.4754 G Loss: 4.5595 Throughput: 5322.72 images/sec
Epoch [1/20] Batch [200/1072] D Loss: 0.0471 G Loss: 4.4322 Throughput: 5319.66 images/sec
Epoch [1/20] Batch [300/1072] D Loss: 3.0825 G Loss: 10.7999 Throughput: 5181.65 images/sec
Epoch [1/20] Batch [400/1072] D Loss: 1.3372 G Loss: 3.2086 Throughput: 3927.60 images/sec
Epoch [1/20] Batch [500/1072] D Loss: 0.4119 G Loss: 2.1206 Throughput: 5034.52 images/sec
Epoch [1/20] Batch [600/1072] D Loss: 1.3683 G Loss: 1.0333 Throughput: 5073.82 images/sec
Epoch [1/20] Batch [700/1072] D Loss: 1.0848 G Loss: 1.1134 Throughput: 5264.68 images/sec
Epoch [1/20] Batch [800/1072] D Loss: 1.2182 G Loss: 1.0841 Throughput: 4439.74 images/sec
Epoch [1/20] Batch [900/1072] D Loss: 1.2657 G Loss: 0.9321 Throughput: 4193.98 images/sec
Epoch [1/20] Batch [1000/1072] D Loss: 1.3035 G Loss: 0.7921 Throughput: 3840.17 images/sec


In [15]:
# Step 6: Save the trained models
torch.save(generator.state_dict(), "generator.pth")
torch.save(discriminator.state_dict(), "discriminator.pth")

